# The AI Chef That Can't Poison Anyone 🍳

*In-memory RDF reasoning for Neo4j — the full demo, runnable top to bottom.*

Your AI sous-chef proposed swapping margarine for **butter** in the béchamel. Fluent, plausible, delicious — and it silently turns your *vegan* lasagna into a mislabeled product with an **undeclared allergen** (EU 1169: that's a recall, not a bad review).

**LLMs guess. Ontologies know.** This notebook builds the guardrail: scope with Cypher, shape with a template, reason with Jena, validate with SHACL — `project → reason → drop`, the GDS mental model applied to meaning.

**Setup** — the next three cells are idempotent: install the client, build the sidecar if the JAR is missing, launch it if nothing answers on `:7475`.

Create a sibling `.env` (see [`.env.example`](.env.example)) with `NEO4J_URI` / `NEO4J_USER` / `NEO4J_PASSWORD` / `NEO4J_DATABASE` pointing at a Neo4j instance — the notebook scopes and fetches its rows over Bolt, seeding the tiny demo graph on first run.

In [1]:
%pip install -q "../n20s-python[neo4j]" python-dotenv
# (plain install: works with any pip; use -e only with pip >= 21.3)
# On Colab instead:
# %pip install -q "n20s-client[neo4j] @ git+https://github.com/halftermeyer/neo4j-n20s#subdirectory=n20s-python" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
%%bash
# build the sidecar fat JAR if it isn't there yet
ls ../n20s-server/target/n20s-server-*.jar 2>/dev/null \
  || (cd .. && mvn -pl n20s-server -am package -q -DskipTests)

../n20s-server/target/n20s-server-1.1.0-SNAPSHOT.jar


In [3]:
%%bash
# launch n20s-server on :7475 unless something already answers
curl -s localhost:7475/version >/dev/null 2>&1 || {
  PORT=7475 nohup java -jar ../n20s-server/target/n20s-server-*.jar \
    >/tmp/n20s-server.log 2>&1 &
  sleep 3
}
curl -s localhost:7475/version

{"version":"server-dev","jenaVersion":"<development>"}

In [4]:
import json, os

try:
    from n20s import N20s
except ImportError:  # running from the repo without pip install
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd().parent / 'n20s-python'))
    from n20s import N20s

if not hasattr(N20s('http://x').graph, 'validateWithRules'):
    raise RuntimeError('Stale n20s client loaded in this kernel — restart the kernel'
                       ' (the pip cell installs the current version).')

try:  # sibling .env: NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD / NEO4J_DATABASE / N20S_URL
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.environ.get('NEO4J_URI'):
    raise RuntimeError('Set NEO4J_URI (e.g. in a sibling .env — see .env.example)')

from neo4j import GraphDatabase
driver = GraphDatabase.driver(
    os.environ['NEO4J_URI'],
    auth=(os.environ.get('NEO4J_USER', 'neo4j'), os.environ.get('NEO4J_PASSWORD', '')))

n20s = N20s(os.environ.get('N20S_URL', 'http://localhost:7475'), driver=driver,
            database=os.environ.get('NEO4J_DATABASE'))
n20s.version()

{'version': 'server-dev', 'jenaVersion': '<development>'}

## The cast

An ordinary recipe graph — nothing semantic about it. In Neo4j it would be:

```cypher
CREATE (lasagna:Recipe {id:'lasagna', name:'Lasagna', diet:'vegan', declared:['gluten']}),
       (bechamel:Recipe:Sauce {id:'bechamel', name:'Béchamel'}),
       (butter:Ingredient:Dairy {id:'butter', name:'Butter', allergens:['milk']}),
       (flour:Ingredient {id:'flour', name:'Wheat flour', allergens:['gluten']}),
       (tomato:Ingredient:Vegetable {id:'tomato', name:'Tomato', allergens:[]}),
       (lasagna)-[:CONTAINS]->(bechamel), (lasagna)-[:CONTAINS]->(tomato),
       (bechamel)-[:CONTAINS]->(butter),  (bechamel)-[:CONTAINS]->(flour);
```

Spot the two bugs the agent introduced: the lasagna still **claims vegan**, and its label still **declares only gluten**.

In [5]:
# Seed the cast into Neo4j once if it's missing (never touches an existing graph)
with driver.session(database=os.environ.get('NEO4J_DATABASE')) as s:
    n = s.run("MATCH (r:Recipe {id:'lasagna'}) RETURN count(r) AS c").single()['c']
    if n == 0:
        s.run('''
            CREATE (lasagna:Recipe {id:'lasagna', name:'Lasagna', diet:'vegan',
                                    declared:['gluten']}),
                   (bechamel:Recipe:Sauce {id:'bechamel', name:'Béchamel'}),
                   (butter:Ingredient:Dairy {id:'butter', name:'Butter', allergens:['milk']}),
                   (flour:Ingredient {id:'flour', name:'Wheat flour', allergens:['gluten']}),
                   (tomato:Ingredient:Vegetable {id:'tomato', name:'Tomato', allergens:[]}),
                   (lasagna)-[:CONTAINS]->(bechamel), (lasagna)-[:CONTAINS]->(tomato),
                   (bechamel)-[:CONTAINS]->(butter),  (bechamel)-[:CONTAINS]->(flour)''')
        print('Seeded the cast into Neo4j 🍝')
    else:
        print('Cast already present in Neo4j')

Cast already present in Neo4j


## Act I — Templates: the mapping is data, not code

Template-driven projection: a declarative template (with R2RML-style IRI templates) turns graph entities into triples. The **label map** is a reviewable alignment table (labels → ontology classes; unmapped labels drop out). **Lists fan out** — `allergens: ['milk']` emits one triple per element. **Missing data skips quietly** — béchamel has no `diet`, so that pattern just doesn't fire.

In [6]:
FOOD = 'http://example.org/food#'

food_template = {
    'subject': FOOD + '{id}',
    'triples': [
        {'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
         'object': {'from': '_labels', 'map': {
             'Ingredient': FOOD + 'Ingredient',
             'Dairy':      FOOD + 'Dairy',
             'Vegetable':  FOOD + 'Vegetable',
             'Recipe':     FOOD + 'Recipe',
             'Sauce':      FOOD + 'Sauce'}},
         'kind': 'iri'},
        {'predicate': FOOD + 'hasAllergen',
         'object': FOOD + 'allergen_{allergens}', 'kind': 'iri'},
        {'predicate': FOOD + 'claimsDiet',
         'object': FOOD + 'diet_{diet}', 'kind': 'iri'},
        {'predicate': FOOD + 'declaresAllergen',
         'object': FOOD + 'allergen_{declared}', 'kind': 'iri'},
        {'predicate': 'http://www.w3.org/2000/01/rdf-schema#label',
         'object': '{name}'},
    ],
}

Now **scope and project** — Cypher's moment. The proposal touches one dish, so we project the *lasagna's subtree* (`-[:CONTAINS*0..]->`), not the menu, into a session-scoped check graph.

In [7]:
G = 'check_lasagna'
n20s.graph.drop(G)

# the client runs the scoping Cypher over Bolt and converts the nodes to rows
n20s.graph.projectTemplate(
    G, food_template,
    cypher='MATCH (:Recipe {id:$id})-[:CONTAINS*0..]->(n) RETURN n',
    params={'id': 'lasagna'})

{'graphName': 'check_lasagna',
 'rows': 5,
 'tripleCount': 17,
 'status': 'projected'}

## Act II — Edges become triples too

A second template maps **relationship types → predicates** — the second alignment table, symmetric with labels → classes. Entities are addressed by **name** (`{s}`, `{r}`, `{t}`), so a template can never silently reverse an edge.

In [8]:
contains_template = {
    'subject': FOOD + '{s.id}',
    'triples': [
        {'predicate': {'from': 'r._type', 'map': {'CONTAINS': FOOD + 'contains'}},
         'object': FOOD + '{t.id}', 'kind': 'iri'},
    ],
}

# Cypher's * walks the tree; multi-column records become named {s, r, t} rows
n20s.graph.projectTemplate(
    G, contains_template,
    cypher='''MATCH p = (:Recipe {id:$id})-[:CONTAINS*]->()
              UNWIND relationships(p) AS r
              RETURN DISTINCT startNode(r) AS s, r, endNode(r) AS t''',
    params={'id': 'lasagna'}, ifExists='append')

{'graphName': 'check_lasagna',
 'rows': 4,
 'tripleCount': 4,
 'status': 'projected'}

## Act III — Two triples of knowledge

The entire ontology this demo needs: *dairy things are animal products*, and *containment is transitive*.

In [9]:
ontology = '''
@prefix food: <http://example.org/food#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
food:Dairy    rdfs:subClassOf food:AnimalProduct .
food:contains a owl:TransitiveProperty .
'''
n20s.graph.addTurtle(G, ontology)

{'graphName': 'check_lasagna',
 'triplesBefore': 21,
 'triplesAfter': 23,
 'added': 2}

One domain rule — **allergens propagate up containment** — and we ask with **backward chaining**: the rule recurses *during the query*, nothing is materialized. Watch the milk arrive from two hops down (*lasagna → béchamel → butter*).

In [10]:
propagation_rule = (
    '[allergenPropagation: (?r http://example.org/food#contains ?i)'
    ' (?i http://example.org/food#hasAllergen ?a)'
    ' -> (?r http://example.org/food#hasAllergen ?a)]')

n20s.graph.queryWithRules(
    G,
    'SELECT ?a WHERE { <http://example.org/food#lasagna>'
    ' <http://example.org/food#hasAllergen> ?a }',
    propagation_rule)

[{'a': 'http://example.org/food#allergen_gluten'},
 {'a': 'http://example.org/food#allergen_milk'}]

And what should a vegan fear? `AnimalProduct` **exists nowhere in the property graph** — no label, no property. It exists in one ontology triple, and the OWL profile applies it at query time.

In [11]:
n20s.graph.query(
    G,
    'SELECT ?x WHERE { <http://example.org/food#lasagna>'
    ' <http://example.org/food#contains> ?x .'
    ' ?x a <http://example.org/food#AnimalProduct> }',
    profile='OWL_MICRO')

[{'x': 'http://example.org/food#butter'}]

**Show your work.** An answer an auditor can't inspect is just a better-dressed guess — so every entailment carries its derivation. `explain` re-derives the statement inside the window with derivation logging on and returns the proof tree: which rule fired, on which premises, down to the asserted facts. Watch the milk climb the containment chain:

In [12]:
for s in n20s.graph.explain(
        G,
        FOOD + 'lasagna', FOOD + 'hasAllergen', FOOD + 'allergen_milk',
        rules=propagation_rule):
    kind = s['kind'] + (f" · rule {s['rule']}" if s.get('rule') else '')
    triple = ' '.join(x.split('#')[-1] for x in (s['subject'], s['predicate'], s['object']))
    print('    ' * s['depth'] + f"[{kind}] {triple}")

[derived · rule allergenPropagation] lasagna hasAllergen allergen_milk
    [asserted] lasagna contains bechamel
    [derived · rule allergenPropagation] bechamel hasAllergen allergen_milk
        [asserted] bechamel contains butter
        [asserted] butter hasAllergen allergen_milk


The receipt — after both questions, the graph still holds **exactly what was projected**. The answers were computed, returned, and gone.

In [13]:
n20s.graph.list()

[{'graphName': 'check_lasagna', 'tripleCount': 23}]

## Act IV — SHACL renders the verdict

`validateWithRules` runs SHACL over **ephemeral inference**: the OWL profile and the propagation rule are applied *inside the call* — the shapes see every entailment, the verdict comes back, and nothing is ever written to the graph. Same contract as the backward queries above.

(Forward chaining — `infer()` / `inferWithRules()` — still has its place: materialize once when you'll run *many* queries over the same projection, or to export the entailed graph via `toTurtle()`. A one-shot check doesn't need it.)

In [14]:
shapes = '''
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix food: <http://example.org/food#> .
food:VeganShape a sh:NodeShape ;
    sh:targetSubjectsOf food:claimsDiet ;
    sh:sparql [
        sh:message "Claims vegan but contains an animal product" ;
        sh:select "PREFIX food: <http://example.org/food#> SELECT $this ?value WHERE { $this food:claimsDiet food:diet_vegan . $this food:contains ?value . ?value a food:AnimalProduct . }" ;
    ] .
food:DeclarationShape a sh:NodeShape ;
    sh:targetSubjectsOf food:declaresAllergen ;
    sh:sparql [
        sh:message "EU 1169: allergen present but not declared on the label" ;
        sh:select "PREFIX food: <http://example.org/food#> SELECT $this ?value WHERE { $this food:hasAllergen ?value . FILTER NOT EXISTS { $this food:declaresAllergen ?value } }" ;
    ] .
'''
n20s.graph.addTurtle(G, shapes)

for v in n20s.graph.validateWithRules(G, propagation_rule, profile='OWL_MICRO'):
    print(f"[{v['severity']}] {v['focusNode'].split('#')[-1]}: {v['message']}")
    print(f"            → {v['value'].split('#')[-1]}")

[Violation] lasagna: EU 1169: allergen present but not declared on the label
            → allergen_milk
[Violation] lasagna: Claims vegan but contains an animal product
            → butter


Both bugs the agent introduced, caught — the vegan lie by ontology-driven inference, the undeclared milk by regulation-shaped constraint. The agent can now regenerate with these violations in context: margarine stays, or the label changes. **Nothing ships that the rules forbid.**

And the final receipt — the projection plus the shapes, exactly as asserted. **The verdict never touched the graph.**

In [15]:
n20s.graph.list()

[{'graphName': 'check_lasagna', 'tripleCount': 33}]

## Drop — the check was ephemeral

The graph of record never changed. Every step above is replayable in front of an auditor.

In [16]:
n20s.graph.drop(G)

{'graphName': 'check_lasagna', 'status': 'dropped'}

---

**Learn more**

- Repo: [github.com/halftermeyer/neo4j-n20s](https://github.com/halftermeyer/neo4j-n20s) — plugin + server + Python client
- Template semantics, every rule as *row + template → triples*: [TEMPLATES.md](https://github.com/halftermeyer/neo4j-n20s/blob/main/TEMPLATES.md)
- Agent integration guide: [BEST_PRACTICES.md](https://github.com/halftermeyer/neo4j-n20s/blob/main/BEST_PRACTICES.md)

*Scope with Cypher. Reason with ontologies. Verify everything.*